# Demo 0 — Preparing photographs

ReMatch never matches a raw photograph. Between the camera and the matcher there
is a preparation stage, and **how many steps it has depends on the species**.
This notebook runs both routes side by side on real images.

| | Balearic wall lizard | Plains zebra |
|---|---|---|
| **1a. Segment** | YOLO, fine-tuned on lizard bellies | Grounded-SAM, prompted with the word `"zebra"` |
| **1b. Extract pattern** | **Yes** — isolate the scale mosaic | **No** — the stripes are the pattern |
| Matching runs on | `data/images-pattern/` | `data/images-segmented/` |

The difference is not arbitrary. A lizard is identified by the *mosaic of ventral
scales*, a fine texture that a second pass has to lift out of the photograph
before it can be matched. A zebra is identified by its stripes, which are already
the dominant structure in the segmented animal — a pattern-extraction step would
only throw information away. Seals and whale sharks are like the zebra;
fingerprints are like the lizard.

Which segmenter to use is a separate question, and a cheaper one. Grounded-SAM
needs no training at all — you tell it what animal to look for in plain English —
so it is what you should try first on a new species. YOLO is worth its annotation
cost only once you have enough labelled images to fine-tune on, and the lizards
are the case where that paid off.

## Setup

In [ ]:
import sys, time, warnings
from pathlib import Path

import cv2, numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd()))          # run from the repository root

import params.image_preparation_params as prep_params
import utils.demo_utils as D
import utils.image_preparation_utils as U
from utils.utils import read_transparent_img

D.set_seed(0)
DEVICE = D.device()

LIZARD_RAW     = Path("data/raw-demo/lizard")            # 4 raw photographs
LIZARD_PATTERN = Path("data/raw-demo/lizard-pattern")    # what step 1b produced
ZEBRA_RAW      = Path("data/images")                     # raw, one dir per animal

OUT = Path("results/demo-0"); OUT.mkdir(parents=True, exist_ok=True)

print("device:", DEVICE)
print("lizards:", len(list(LIZARD_RAW.glob("*.jpg"))), "raw photographs")
print("zebras :", len(list(ZEBRA_RAW.glob("zebra_*/*.jpg"))), "raw photographs")


def strip(axes):
    for ax in np.ravel(axes):
        ax.axis("off")


def row(paths, titles, suptitle, loader=None, cmap=None):
    """One row of images, sized to however many there are."""
    n = len(paths)
    fig, axes = plt.subplots(1, n, figsize=(3.6 * n, 3.6), squeeze=False)
    for ax, p, t in zip(axes[0], paths, titles):
        img = loader(p) if loader else cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        ax.imshow(img, cmap=cmap)
        ax.set_title(t, fontsize=8)
    strip(axes)
    plt.suptitle(suptitle, y=1.04)
    plt.tight_layout(); plt.show()

### Do you have what this notebook needs?

Unlike `demo-1` and `demo-2`, which run on the pattern crops bundled in `data/`,
this notebook actually runs the segmenters — so it needs their weights, and none
of them are in the repository. The next cell tells you which are missing and
where to get them, rather than letting you find out halfway down the page.

In [ ]:
REQUIRED = {
    "models/yolo-segmentation.pt": (
        "YOLO, fine-tuned on lizard bellies (route 1)",
        "https://huggingface.co/roberalcaraz/lizard-body-segmentation",
    ),
    "models/groundingdino_swint_ogc.pth": (
        "GroundingDINO, text-prompted detection (route 2)",
        "https://github.com/IDEA-Research/GroundingDINO#luggage-checkpoints",
    ),
    "models/GroundingDINO_SwinT_OGC.py": (
        "GroundingDINO config (ships with the repository)",
        "already in models/ - restore it from git if it is gone",
    ),
    "models/sam_vit_h_4b8939.pth": (
        "SAM ViT-H, turns boxes into masks (route 2)",
        "https://github.com/facebookresearch/segment-anything#model-checkpoints",
    ),
}

missing = [(p, why, url) for p, (why, url) in REQUIRED.items() if not Path(p).exists()]

try:
    import groundingdino  # noqa: F401
    dino_ok = True
except ImportError:
    dino_ok = False

for p, (why, _) in REQUIRED.items():
    print(f"  {'ok ' if Path(p).exists() else 'MISSING'}  {p:38} {why}")
print(f"  {'ok ' if dino_ok else 'MISSING'}  {'groundingdino (package)':38} needed by route 2")

if missing or not dino_ok:
    print("\nBefore running the rest of this notebook:")
    for p, _, url in missing:
        print(f"  - put {Path(p).name} in models/  <-  {url}")
    if not dino_ok:
        print('  - uv pip install --no-build-isolation \\')
        print('      "git+https://github.com/IDEA-Research/GroundingDINO.git" "transformers<5"')
    print("\nRoute 1 needs the YOLO weights; route 2 needs the other two plus the")
    print("package. Whichever you have, that route will run - skip the other.")
else:
    print("\nEverything is here. Run on.")

---
# Route 1 — Balearic wall lizard

## The raw photographs

A lizard held belly-up against whatever was to hand: a notebook, a knee, the
ground. The identity is in the scales of the belly, so everything else in the
frame — the hand, the background, the animal's own head and legs — is noise the
preparation stage has to remove.

In [ ]:
lizard_raws = sorted(LIZARD_RAW.glob("*.jpg"))
row(lizard_raws, [p.stem for p in lizard_raws], "Raw photographs, straight from the camera")
for p in lizard_raws:
    h, w = cv2.imread(str(p)).shape[:2]
    print(f"  {p.name:22} {w}x{h}  {p.stat().st_size/1024:.0f} KB")

## Step 1a — Segment with YOLO

`YOLO_SEGMENTATION_MODEL` is a YOLO instance-segmentation model fine-tuned on
annotated lizard bellies, so it detects one class and one only. It returns a mask
rather than a box, and the pipeline uses it to crop the animal, cut the
background out through the alpha channel, and rotate the belly upright so that
later stages see a consistent orientation.

This needs `models/yolo-segmentation.pt` — see the README.

In [ ]:
LIZARD_SEG = OUT / "lizard-segmented"; LIZARD_SEG.mkdir(parents=True, exist_ok=True)

t0 = time.time()
U.YOLO_segmentation(
    prep_params.YOLO_SEGMENTATION_MODEL,
    str(LIZARD_RAW),
    str(LIZARD_SEG),
    str(OUT),
)
print(f"\nsegmented {len(lizard_raws)} photographs in {time.time()-t0:.0f}s")
print("\nMessages about 'more than 1 detection' or 'low confidence' are the")
print("pipeline flagging images for review - it keeps them, but logs them to")
print(f"{OUT}/ so you can check them by eye before matching.")

In [ ]:
segs = [LIZARD_SEG / f"{p.stem}.png" for p in lizard_raws]
row(segs, [p.stem for p in segs],
    "After YOLO: belly isolated, background transparent, rotated upright",
    loader=lambda p: read_transparent_img(str(p)), cmap="gray")

for p in segs:
    im = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    covered = (im[:, :, 3] > 0).mean()
    print(f"  {p.name:22} {im.shape[1]}x{im.shape[0]}  mask covers {covered:.0%} of the crop")

## Step 1b — Extract the scale pattern

This is the step the zebras do not need. SAM proposes a mask for every scale,
structured edge detection traces their boundaries, and the result is the *mosaic*
— each scale reduced to its outline, with the animal's colour, lighting and skin
tone discarded. What survives is the geometry, which is what GlueStick matches on.

**This step cannot run from a clean install right now.**
`utils.image_preparation_utils.extract_pattern_from_images` imports `tensorflow`
and `ISR` (for a denoising super-resolution pass), neither of which is declared
in `requirements.txt` — and `ISR` pins `tensorflow` to 1.13.1 or 2.0.0, versions
that have no wheels for any Python this repository supports. So the cell below
shows the crops the published pipeline produced, rather than recomputing them.
The images in `data/images-pattern/`, which `demo-1` trains on, were made this
way.

In [ ]:
pats = sorted(LIZARD_PATTERN.glob("*.png"))
row(pats, [p.stem for p in pats],
    "Step 1b output: the scale mosaic, as edges (precomputed - see above)",
    loader=lambda p: cv2.imread(str(p), cv2.IMREAD_GRAYSCALE), cmap="gray")

try:
    import tensorflow, ISR          # noqa: F401
    print("tensorflow and ISR are importable - you can run step 1b yourself:")
    print("  U.extract_pattern_from_images(str(LIZARD_SEG), str(OUT/'lizard-pattern'),")
    print("      str(OUT), prep_params.SAM_CHECKPOINT_PATH, prep_params.EDGE_NMS_PATH, DEVICE)")
except ImportError as e:
    print(f"step 1b cannot run here: {e}")
    print("showing the published crops instead.")

## The three stages together

Read a column top to bottom: what the camera saw, what the segmenter kept, and
what the matcher finally sees.

In [ ]:
fig, ax = plt.subplots(3, len(lizard_raws), figsize=(3.6 * len(lizard_raws), 10), squeeze=False)
for c, p in enumerate(lizard_raws):
    ax[0][c].imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
    ax[0][c].set_title(f"1. raw\n{p.stem}", fontsize=8)
    ax[1][c].imshow(read_transparent_img(str(LIZARD_SEG / f"{p.stem}.png")), cmap="gray")
    ax[1][c].set_title("2. segmented (YOLO)", fontsize=8)
    ax[2][c].imshow(cv2.imread(str(LIZARD_PATTERN / f"{p.stem}.png"), cv2.IMREAD_GRAYSCALE), cmap="gray")
    ax[2][c].set_title("3. pattern (matched on)", fontsize=8)
strip(ax)
plt.suptitle("Lizard: raw -> segmented -> pattern", y=1.01)
plt.tight_layout(); plt.show()

---
# Route 2 — Plains zebra

## The raw photographs

Field photographs: the animal at a distance, vegetation, other zebras in frame.
Nothing was annotated to make this work.

In [ ]:
# One photograph from each of three different animals, not three of the same.
zebra_raws = [sorted(d.glob("*.jpg"))[0]
              for d in sorted(p for p in ZEBRA_RAW.iterdir() if p.is_dir())[:3]]
row(zebra_raws, [f"{p.parent.name}\n{p.stem}" for p in zebra_raws],
    "Raw field photographs")

## Step 1a — Segment with Grounded-SAM

Two models in series and no training whatsoever. GroundingDINO reads the word
`"zebra"` and proposes boxes for anything matching it; SAM turns the best box
into a pixel mask. Changing `CLASSES` to `["seal"]` is the entire cost of moving
to another species — which is why this is the route to try first, and why
`demo-3` uses it.

Needs `models/groundingdino_swint_ogc.pth` and `models/sam_vit_h_4b8939.pth`,
plus the GroundingDINO package. It is slow on CPU: SAM's ViT-H encoder runs on
every photograph.

In [ ]:
ZEBRA_SEG = OUT / "zebra-segmented"; ZEBRA_SEG.mkdir(parents=True, exist_ok=True)

t0 = time.time()
gsam = D.build_grounded_sam(DEVICE)
written, flagged = D.segment_images(gsam, zebra_raws, ZEBRA_SEG, ["zebra"], DEVICE)
del gsam; D.empty_device_cache(DEVICE)
print(f"\nsegmented {len(written)} photographs in {time.time()-t0:.0f}s")
for p, why in flagged:
    print(f"  ! {Path(p).name}: {why}")

In [ ]:
fig, ax = plt.subplots(2, len(zebra_raws), figsize=(4.4 * len(zebra_raws), 7), squeeze=False)
for c, p in enumerate(zebra_raws):
    ax[0][c].imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
    ax[0][c].set_title(f"1. raw\n{p.parent.name}", fontsize=8)
    ax[1][c].imshow(D.load_gray(ZEBRA_SEG / f"{p.stem}.png"), cmap="gray")
    ax[1][c].set_title("2. segmented (Grounded-SAM) - matched on", fontsize=8)
strip(ax)
plt.suptitle("Zebra: raw -> segmented. There is no third stage.", y=1.01)
plt.tight_layout(); plt.show()

---
## Choosing a route for your own species

1. **Always start with Grounded-SAM.** It costs nothing to try: name your animal
   in `CLASSES` and look at the masks. On the zebras above it needed no tuning at
   all.
2. **Only fine-tune YOLO if Grounded-SAM keeps failing** — if it grabs the
   background instead of the animal, or cannot separate two animals in frame.
   That is a few hundred annotated images of work, so make it earn its place.
3. **Skip pattern extraction unless the identity is a fine texture** the
   segmented animal does not already show. Stripes, rings and spots do not need
   it; scale mosaics and dermal ridges do. Of the five datasets in the paper,
   only the lizards and the fingerprints use it.
4. **Look at the output before you match.** Matching is the expensive step and a
   bad mask wastes all of it. Both `demo-3` and the pipeline flag low-confidence
   detections rather than dropping them silently, so there is a list to review.

Once the images are prepared, [`demo-1`](demo-1-training.ipynb) trains a model on
them and [`demo-3`](demo-3-new-species.ipynb) does the whole thing end to end for
a species that needs no pattern extraction.